In [16]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score
from itertools import product
from datetime import datetime
import random
import csv


In [17]:
# ---------------- 기본 설정 ----------------
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
# ---------------- Dataset 정의 ----------------
class PostureDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.data = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(image_path).convert("RGB")
        label = row["class_id"]
        if self.transform:
            image = self.transform(image)
        return image, label

In [19]:
# ---------------- 모델 구성 ----------------
def get_backbone(name="mobilenet_v3_large"):
    model = models.mobilenet_v3_large(pretrained=True)
    in_features = model.classifier[0].in_features
    model.classifier = nn.Identity()
    return model, in_features

def get_mlp_head(name, in_features):
    if name == "mini":
        return nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Linear(in_features, 1)
        )
    elif name == "simple":
        return nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Linear(in_features, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    elif name == "strong":
        return nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Linear(in_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 1)
        )
    else:
        return nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

def get_augment(level):
    if level == "none":
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])
    elif level == "medium":
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.transforms.Compose([
            transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), # 이미지 랜덤 크롭 후 224x224로 리사이즈 (다양한 스케일 학습)
            transforms.RandomHorizontalFlip(p=0.5),             # 50% 확률로 좌우 반전
            transforms.RandomVerticalFlip(p=0.2),               # 20% 확률로 상하 반전 (데이터 특성에 따라 조절 필요)
            transforms.RandomRotation(degrees=30),              # -30도에서 +30도 사이로 랜덤 회전
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1), # 색상, 밝기, 대비 등 랜덤 변경
            transforms.RandomAffine(degrees=10, translate=(0.15, 0.15), scale=(0.9, 1.1), shear=10), # 이동, 스케일, 전단 변형
            # transforms.RandomErasing(p=0.2, scale=(0.02, 0.1), ratio=(0.3, 3.3)), # 선택 사항: 이미지 일부를 랜덤하게 가림 (강력한 증강)
            transforms.ToTensor(),                              # PIL Image를 PyTorch Tensor로 변환
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # ImageNet 통계 기반 정규화
        ])

In [20]:
# ---------------- 학습/검증 루프 ----------------
def train_epoch(loader, model, criterion, optimizer, is_train=True):
    model.train() if is_train else model.eval()
    preds, labels = [], []
    running_loss = 0.0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.float().unsqueeze(1).to(device)

        if is_train:
            optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, targets)

        if is_train:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds_batch = (torch.sigmoid(outputs).detach().cpu().numpy() > 0.5).astype(int)
        labels_batch = targets.cpu().numpy().astype(int)

        preds.extend(preds_batch)
        labels.extend(labels_batch)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)

    return running_loss / len(loader.dataset), acc, f1

In [ ]:
# ---------------- 실험 실행 ----------------
def run_experiment(writer, mlp_type, aug_level, use_scheduler, batch_size, weight_decay):
    print(f"\n🔧 Experiment: MLP={mlp_type}, AUG={aug_level}, SCH={use_scheduler}, BS={batch_size}")
    
    # 데이터셋 로딩
    train_df = pd.read_csv("../dataset-modification/train_pose_parsed.csv")[["filename", "class_id"]]
    val_df = pd.read_csv("../dataset-modification/valid_pose_parsed.csv")[["filename", "class_id"]]
    train_df = train_df.groupby("filename")["class_id"].min().reset_index()
    val_df = val_df.groupby("filename")["class_id"].min().reset_index()

    class_weights = compute_class_weight('balanced', classes=np.unique(train_df["class_id"]), y=train_df["class_id"])
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights_tensor[1])

    train_transform = get_augment(aug_level)
    val_transform = get_augment("none")

    train_dataset = PostureDataset(train_df, "../dataset-modification/train-visualized/images", train_transform)
    val_dataset = PostureDataset(val_df, "../dataset-modification/valid-visualized/images", val_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    # 모델 구성
    backbone, in_features = get_backbone()
    mlp_head = get_mlp_head(mlp_type, in_features)
    model = nn.Sequential(backbone, mlp_head).to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5) if use_scheduler else None

    for epoch in range(10):
            train_loss, train_acc, train_f1 = train_epoch(train_loader, model, criterion, optimizer, True)
            val_loss, val_acc, val_f1 = train_epoch(val_loader, model, criterion, optimizer, False)
            if scheduler:
                scheduler.step()

            # 매 epoch마다 결과만 출력
            print(f"[{epoch+1}/10] "
                f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f} | "
                f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")

            # CSV 로그 기록
            writer.writerow([mlp_type, aug_level, use_scheduler, batch_size, weight_decay, epoch+1,
                            train_loss, train_acc, train_f1, val_loss, val_acc, val_f1])


In [ ]:
if __name__ == "__main__":
    mlp_types = ['mini', 'simple', 'default', 'strong']
    augment_levels = ['none', 'medium', 'strong']
    use_schedulers = [False, True]
    batch_sizes = [16, 32]
    weight_decays = [0.0, 1e-4]  # 사용 안함 vs 사용함

    log_filename = f"experiment_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

    # ✅ writer 생성과 실험 실행을 같은 with 블록 안에서 수행해야 함
    with open(log_filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "MLP", "Augment", "Scheduler", "BatchSize", "WeightDecay", "Epoch",
            "TrainLoss", "TrainAcc", "TrainF1", "ValLoss", "ValAcc", "ValF1"
        ])

        for mlp, aug, sch, bs, wd in product(mlp_types, augment_levels, use_schedulers, batch_sizes, weight_decays):
            run_experiment(writer, mlp, aug, sch, bs, wd)

    print(f"\n✅ 모든 실험 완료. 결과가 {log_filename} 에 저장되었습니다.")



🔧 Experiment: MLP=mini, AUG=none, SCH=False, BS=16


c:\Users\main\miniconda3\envs\PoseDetect-env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\main\miniconda3\envs\PoseDetect-env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[1/15] Train Loss: 0.5892, Acc: 0.7179, F1: 0.6730 | Val Loss: 0.4500, Acc: 0.8286, F1: 0.8039
[2/15] Train Loss: 0.3571, Acc: 0.8628, F1: 0.8324 | Val Loss: 0.3922, Acc: 0.8571, F1: 0.8355
[3/15] Train Loss: 0.2601, Acc: 0.9179, F1: 0.8963 | Val Loss: 0.4195, Acc: 0.8414, F1: 0.8128
[4/15] Train Loss: 0.1931, Acc: 0.9372, F1: 0.9201 | Val Loss: 0.4332, Acc: 0.8529, F1: 0.8227
[5/15] Train Loss: 0.1662, Acc: 0.9438, F1: 0.9285 | Val Loss: 0.4293, Acc: 0.8543, F1: 0.8235
[6/15] Train Loss: 0.1360, Acc: 0.9570, F1: 0.9445 | Val Loss: 0.4900, Acc: 0.8600, F1: 0.8339
[7/15] Train Loss: 0.1159, Acc: 0.9605, F1: 0.9491 | Val Loss: 0.5025, Acc: 0.8643, F1: 0.8342
[8/15] Train Loss: 0.1097, Acc: 0.9686, F1: 0.9595 | Val Loss: 0.5143, Acc: 0.8557, F1: 0.8303
[9/15] Train Loss: 0.1056, Acc: 0.9678, F1: 0.9587 | Val Loss: 0.5216, Acc: 0.8471, F1: 0.8106
[10/15] Train Loss: 0.1037, Acc: 0.9667, F1: 0.9575 | Val Loss: 0.5144, Acc: 0.8529, F1: 0.8239

🔧 Experiment: MLP=mini, AUG=none, SCH=False, BS=

c:\Users\main\miniconda3\envs\PoseDetect-env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\main\miniconda3\envs\PoseDetect-env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[1/15] Train Loss: 0.5828, Acc: 0.7257, F1: 0.6805 | Val Loss: 0.4677, Acc: 0.8086, F1: 0.7970
[2/15] Train Loss: 0.3534, Acc: 0.8690, F1: 0.8384 | Val Loss: 0.4070, Acc: 0.8571, F1: 0.8288
[3/15] Train Loss: 0.2397, Acc: 0.9210, F1: 0.9000 | Val Loss: 0.3923, Acc: 0.8600, F1: 0.8262
[4/15] Train Loss: 0.1892, Acc: 0.9415, F1: 0.9254 | Val Loss: 0.4525, Acc: 0.8543, F1: 0.8277
[5/15] Train Loss: 0.1764, Acc: 0.9465, F1: 0.9313 | Val Loss: 0.4759, Acc: 0.8600, F1: 0.8322
[6/15] Train Loss: 0.1353, Acc: 0.9539, F1: 0.9405 | Val Loss: 0.5279, Acc: 0.8529, F1: 0.8209
[7/15] Train Loss: 0.1352, Acc: 0.9574, F1: 0.9454 | Val Loss: 0.5417, Acc: 0.8614, F1: 0.8330
[8/15] Train Loss: 0.1364, Acc: 0.9578, F1: 0.9457 | Val Loss: 0.5266, Acc: 0.8671, F1: 0.8366
[9/15] Train Loss: 0.0931, Acc: 0.9675, F1: 0.9579 | Val Loss: 0.6156, Acc: 0.8543, F1: 0.8217
[10/15] Train Loss: 0.0900, Acc: 0.9682, F1: 0.9591 | Val Loss: 0.6448, Acc: 0.8614, F1: 0.8271

🔧 Experiment: MLP=mini, AUG=none, SCH=False, BS=

...

[1/15] Train Loss: 0.6437, Acc: 0.7013, F1: 0.6487 | Val Loss: 0.6186, Acc: 0.6586, F1: 0.6818
[2/15] Train Loss: 0.4229, Acc: 0.8377, F1: 0.8057 | Val Loss: 0.4567, Acc: 0.8114, F1: 0.7843
[3/15] Train Loss: 0.3404, Acc: 0.8690, F1: 0.8380 | Val Loss: 0.4068, Acc: 0.8486, F1: 0.8251
[4/15] Train Loss: 0.2788, Acc: 0.9020, F1: 0.8762 | Val Loss: 0.4125, Acc: 0.8500, F1: 0.8142
[5/15] Train Loss: 0.2457, Acc: 0.9148, F1: 0.8922 | Val Loss: 0.3900, Acc: 0.8657, F1: 0.8385
[6/15] Train Loss: 0.2001, Acc: 0.9334, F1: 0.9143 | Val Loss: 0.4146, Acc: 0.8586, F1: 0.8260
[7/15] Train Loss: 0.1812, Acc: 0.9380, F1: 0.9207 | Val Loss: 0.4274, Acc: 0.8600, F1: 0.8275
[8/15] Train Loss: 0.1650, Acc: 0.9461, F1: 0.9311 | Val Loss: 0.4396, Acc: 0.8543, F1: 0.8247
[9/15] Train Loss: 0.1564, Acc: 0.9489, F1: 0.9346 | Val Loss: 0.4417, Acc: 0.8700, F1: 0.8428
[10/15] Train Loss: 0.1553, Acc: 0.9489, F1: 0.9346 | Val Loss: 0.4614, Acc: 0.8586, F1: 0.8284

🔧 Experiment: MLP=mini, AUG=medium, SCH=True, BS

c:\Users\main\miniconda3\envs\PoseDetect-env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\main\miniconda3\envs\PoseDetect-env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[1/15] Train Loss: 0.6390, Acc: 0.6831, F1: 0.6364 | Val Loss: 0.6560, Acc: 0.6257, F1: 0.6773
[2/15] Train Loss: 0.4382, Acc: 0.8202, F1: 0.7870 | Val Loss: 0.4400, Acc: 0.8243, F1: 0.8051
[3/15] Train Loss: 0.3337, Acc: 0.8783, F1: 0.8502 | Val Loss: 0.4214, Acc: 0.8614, F1: 0.8319
[4/15] Train Loss: 0.2783, Acc: 0.9086, F1: 0.8852 | Val Loss: 0.4115, Acc: 0.8614, F1: 0.8342
[5/15] Train Loss: 0.2352, Acc: 0.9213, F1: 0.9000 | Val Loss: 0.4107, Acc: 0.8500, F1: 0.8199
[6/15] Train Loss: 0.2022, Acc: 0.9287, F1: 0.9090 | Val Loss: 0.4195, Acc: 0.8471, F1: 0.8152
[7/15] Train Loss: 0.1798, Acc: 0.9438, F1: 0.9277 | Val Loss: 0.4336, Acc: 0.8600, F1: 0.8256
[8/15] Train Loss: 0.1742, Acc: 0.9388, F1: 0.9216 | Val Loss: 0.4368, Acc: 0.8629, F1: 0.8310
[9/15] Train Loss: 0.1649, Acc: 0.9423, F1: 0.9263 | Val Loss: 0.4386, Acc: 0.8557, F1: 0.8231
[10/15] Train Loss: 0.1450, Acc: 0.9523, F1: 0.9385 | Val Loss: 0.4620, Acc: 0.8514, F1: 0.8237

🔧 Experiment: MLP=mini, AUG=strong, SCH=False, B

c:\Users\main\miniconda3\envs\PoseDetect-env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\main\miniconda3\envs\PoseDetect-env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[1/15] Train Loss: 0.6993, Acc: 0.6439, F1: 0.5906 | Val Loss: 0.6218, Acc: 0.7071, F1: 0.6880
[2/15] Train Loss: 0.5601, Acc: 0.7497, F1: 0.7012 | Val Loss: 0.5299, Acc: 0.7643, F1: 0.7273
[3/15] Train Loss: 0.4851, Acc: 0.7989, F1: 0.7537 | Val Loss: 0.5469, Acc: 0.7571, F1: 0.7557
[4/15] Train Loss: 0.4475, Acc: 0.8233, F1: 0.7773 | Val Loss: 0.4513, Acc: 0.8314, F1: 0.8033
[5/15] Train Loss: 0.4020, Acc: 0.8423, F1: 0.8008 | Val Loss: 0.4286, Acc: 0.8386, F1: 0.8101
[6/15] Train Loss: 0.3866, Acc: 0.8563, F1: 0.8173 | Val Loss: 0.4626, Acc: 0.8257, F1: 0.7932
